# Training Classifier

In [1]:
import typing

import sqlalchemy as sa
from datasets import DatasetDict
from sqlalchemy.orm import Session
from sqlalchemy.orm import joinedload, subqueryload
from torch.utils.data import IterableDataset
from transformers import (
    AutoTokenizer,
    PreTrainedTokenizerBase
)

from models import TextClassifierDataset, TextClassifierDatasetItem
from models.datasets import TrainTestSplit
from models.models import TextClassifierModel
from services.app_error import AppError
from services.db import SessionLocal

In [2]:
def get_dataset(session: Session, model_id: int) -> TextClassifierDataset:
    """Retrieve the dataset associated with the model."""
    dataset = session.execute(
        sa.select(TextClassifierDataset)
        .select_from(TextClassifierModel)
        .where(TextClassifierModel.id == model_id)
        .join(TextClassifierDataset, TextClassifierModel.dataset_id == TextClassifierDataset.id)
        .options(
            subqueryload(TextClassifierDataset.labels)
        )
    ).scalar_one_or_none()

    if dataset is None:
        raise AppError(
            error_code='not_found',
            status_code=404,
            details={"id": model_id},
            message=f"Model {model_id} not found"
        )

    return dataset


In [3]:
session = SessionLocal()
dataset = get_dataset(session, 1)
session.close()

dataset

In [4]:
def process_labels(dataset: TextClassifierDataset) -> tuple[list[str], dict[int, str], dict[str, int]]:
    """Process dataset labels into required formats."""
    all_labels = [typing.cast(str, label.label) for label in dataset.labels]
    id2label = {i: label for i, label in enumerate(all_labels)}
    label2id = {label: i for i, label in enumerate(all_labels)}

    return all_labels, id2label, label2id


all_labels, id2label, label2id = process_labels(dataset)
all_labels

['Neutral', 'Positive', 'Extremely Negative', 'Negative', 'Extremely Positive']

In [5]:
class PostgresDataset(IterableDataset):
    def __init__(
            self,
            dataset_id: int,
            split: TrainTestSplit,
            tokenizer: PreTrainedTokenizerBase,
            label2id: dict[str, int],
            batch_size: int = 100
    ):
        self._dataset_id = dataset_id
        self._split = split
        self._tokenizer = tokenizer
        self._batch_size = batch_size
        self._label2id = label2id

    def __iter__(self):
        session = SessionLocal()
        offset = 0

        while True:
            tokenizer = self._tokenizer
            query = sa.text(
                """
SELECT
    tcdi.text_content,
    tcdl.label
FROM
    text_classifier_dataset_items tcdi
LEFT JOIN
    text_classifier_dataset_labels tcdl
    ON tcdi.dataset_label_id = tcdl.dataset_label_id AND tcdi.dataset_id = tcdl.dataset_id
WHERE
    tcdi.dataset_id = :dataset_id
    AND tcdi.split = :split
    AND tcdi.dataset_label_id IS NOT NULL
OFFSET :offset
LIMIT :limit
""".strip()
            )
            rows = session.execute(
                query,
                {
                    "dataset_id": self._dataset_id,
                    "split": self._split.name,
                    "offset": offset,
                    "limit": self._batch_size
                }
            ).all()

            n = len(rows)
            if n == 0:
                break
            for text, label in rows:
                row = tokenizer(text, truncation=True, padding=True)
                row["label"] = self._label2id[label]
                yield row
            offset += n

        session.close()

    def __len__(self):
        session = SessionLocal()

        count = session.execute(
            sa.select(sa.func.count(TextClassifierDatasetItem.id))
            .where(TextClassifierDatasetItem.dataset_id == self._dataset_id)
            .where(TextClassifierDatasetItem.split == self._split)
            .where(TextClassifierDatasetItem.dataset_label_id.isnot(None))
        ).scalar_one_or_none()

        session.close()

        return count if count is not None else 0


def create_dataset_for_split(
        dataset_id: int,
        split: TrainTestSplit,
        tokenizer: PreTrainedTokenizerBase,
        label2id: dict[str, int]
):
    return PostgresDataset(
        dataset_id,
        split,
        tokenizer,
        label2id,
    )

In [6]:
def load_dataset_items(dataset_id: int, tokenizer: PreTrainedTokenizerBase, label2id: dict[str, int]) -> DatasetDict:
    """Load training and testing data from the database."""

    train = create_dataset_for_split(dataset_id, TrainTestSplit.TRAIN, tokenizer, label2id)
    test = create_dataset_for_split(dataset_id, TrainTestSplit.TEST, tokenizer, label2id)

    return DatasetDict({"train": train, "test": test})

In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
    dropout=0.1,
    attention_dropout=0.1
)

def compute_metrics(eval_pred: tuple[np.ndarray, np.ndarray]):
    """Compute evaluation metrics for the model."""
    accuracy = evaluate.load("accuracy")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(predictions=predictions, references=labels)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
hf_dataset = load_dataset_items(dataset.id, tokenizer, label2id)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments(
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=7,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset["train"],
    eval_dataset=hf_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


In [11]:
trainer.model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
